# SPID v4 — $\lambda_{\max}$ Parameter Sweep (Full 14-Matrix Suite)

Reproduces **Remark 4.6** with all 14 benchmarks from the paper at their correct sizes.

Logs dangerous steps ($\lambda_k \geq 2$), ceiling-binding ($\alpha_k$ clipped to $\alpha_{\max}$),
and fallback ($|\mathrm{denom}| < 10^{-12}$) counts per run.

**Runtime note:** Russell 3000 ($n=1000$) will DNF at low $\lambda_{\max}$ values.
Budget ~30–60 min for the full sweep.

In [ ]:
import numpy as np
import scipy.linalg as la
import pandas as pd
import shutil
import time

## 1 — Core Operators

In [ ]:
def proj_U(A):
    """P_U: set diagonal to 1."""
    X = A.copy()
    np.fill_diagonal(X, 1.0)
    return X


def proj_S(A):
    """P_S: clip negative eigenvalues to zero."""
    try:
        w, V = la.eigh(A, subset_by_value=(0.0, np.inf))
        X = V @ np.diag(w) @ V.T
    except ValueError:
        X = np.zeros_like(A)
    return (X + X.T) / 2.0

## 2 — SPID v4 Solver (with self-regulation logging)

In [ ]:
DENOM_GUARD = 1e-12


def spid_v4_sweep(A, lambda_max=5.0, beta=0.5, alpha_min=0.1,
                  tol=1e-5, max_iter=2000):
    """
    SPID v4 with self-regulation counters.

    Returns dict with:
        iters, diag_err, min_eig, converged,
        fallback, ceiling, dangerous, wall_time
    """
    n = A.shape[0]
    large_n = (n >= 500)
    X = A.copy()
    dS = np.zeros((n, n))
    dU = np.zeros((n, n))

    alpha_max = lambda_max / beta
    alpha_fallback = 1.0 / beta

    X_prev = X.copy()
    R_prev = np.zeros((n, n))

    converged = False
    k_final = max_iter

    # Self-regulation counters
    fallback_count  = 0
    ceiling_count   = 0
    dangerous_count = 0

    t0 = time.perf_counter()

    for k in range(1, max_iter + 1):

        # [1] Dykstra cycle
        R = X + dS
        S = proj_S(R)
        dS_new = R - S

        R_U = S + dU
        W = proj_U(R_U)
        dU_new = R_U - W
        T_D = W

        # [2] Ishikawa nesting
        Z_X  = (1 - beta) * X  + beta * T_D
        Z_dS = (1 - beta) * dS + beta * dS_new
        Z_dU = (1 - beta) * dU + beta * dU_new
        R_curr = X - Z_X

        # [3] Alternating BB step size
        used_fallback = False
        if k == 1:
            alpha = alpha_fallback
            used_fallback = True
        else:
            s = X - X_prev
            y = R_curr - R_prev

            tr_SS = np.sum(s * s)
            tr_SY = np.sum(s * y)
            tr_YY = np.sum(y * y)

            if k % 2 == 1:  # BB1
                denom = tr_SY
                if abs(denom) < DENOM_GUARD:
                    alpha = alpha_fallback
                    used_fallback = True
                else:
                    alpha = tr_SS / denom
            else:           # BB2
                denom = tr_YY
                if abs(denom) < DENOM_GUARD:
                    alpha = alpha_fallback
                    used_fallback = True
                else:
                    alpha = tr_SY / denom

        # Clip
        alpha_raw = alpha
        alpha = float(np.clip(alpha, alpha_min, alpha_max))

        # Log events
        if used_fallback:
            fallback_count += 1
        if alpha_raw > alpha_max:
            ceiling_count += 1
        lambda_k = alpha * beta
        if lambda_k >= 2.0:
            dangerous_count += 1

        # [4] Update
        X_next  = X  + alpha * (Z_X  - X)
        dS_next = dS + alpha * (Z_dS - dS)
        dU_next = dU + alpha * (Z_dU - dU)

        # [5] Stopping criterion (matches Algorithm 1)
        if large_n:
            stop = la.norm(X_next - X, "fro") < tol
        else:
            stop = max(la.norm(X_next - S, "fro"),
                       np.max(np.abs(np.diag(X_next) - 1.0))) < tol

        if stop:
            X = X_next
            converged = True
            k_final = k
            break

        # [6] Shift
        X_prev = X
        R_prev = R_curr
        X, dS, dU = X_next, dS_next, dU_next

    wall = time.perf_counter() - t0

    # Feasibility polish
    X_hat = proj_U(proj_S(X))
    diag_err = float(np.max(np.abs(np.diag(X_hat) - 1.0)))
    min_eig  = float(np.min(la.eigvalsh(X_hat)))

    return {
        "iters":     k_final,
        "diag_err":  diag_err,
        "min_eig":   min_eig,
        "converged": converged,
        "fallback":  fallback_count,
        "ceiling":   ceiling_count,
        "dangerous": dangerous_count,
        "wall_time": wall,
    }


def higham_dykstra_iters(A, tol=1e-5, max_iter=2000):
    """Plain Dykstra baseline — returns iteration count."""
    n = A.shape[0]
    large_n = (n >= 500)
    X = A.copy()
    dS = np.zeros_like(A)
    dU = np.zeros_like(A)

    for k in range(1, max_iter + 1):
        R = X + dS
        S = proj_S(R)
        dS_new = R - S

        R_U = S + dU
        W = proj_U(R_U)
        dU_new = R_U - W

        if large_n:
            stop = la.norm(W - X, "fro") < tol
        else:
            stop = max(la.norm(W - S, "fro"),
                       np.max(np.abs(np.diag(W) - 1.0))) < tol

        if stop:
            return k

        X, dS, dU = W, dS_new, dU_new

    return max_iter

## 3 — All 14 Test Matrices (paper sizes)

In [ ]:
def gen_rebonato_jackel(n=100, seed=42):
    np.random.seed(seed)
    i = np.arange(n).reshape(-1, 1); j = np.arange(n).reshape(1, -1)
    G = np.exp(-0.1 * np.abs(i - j))
    G += 0.1 * np.random.randn(n, n)
    return proj_U((G + G.T) / 2.0)

def gen_turkay():
    return np.array([[ 1.00, -0.55, -0.15, -0.10],
                     [-0.55,  1.00,  0.90,  0.90],
                     [-0.15,  0.90,  1.00,  0.90],
                     [-0.10,  0.90,  0.90,  1.00]])

def gen_bhansali():
    return np.array([[ 1.00, -0.50, -0.30, -0.25, -0.70],
                     [-0.50,  1.00,  0.90,  0.30,  0.70],
                     [-0.30,  0.90,  1.00,  0.25,  0.20],
                     [-0.25,  0.30,  0.25,  1.00,  0.75],
                     [-0.70,  0.70,  0.20,  0.75,  1.00]])

def gen_finger():
    return np.array([[ 1.00,  0.18, -0.13, -0.26,  0.19, -0.25, -0.12],
                     [ 0.18,  1.00,  0.22, -0.14,  0.31,  0.16,  0.09],
                     [-0.13,  0.22,  1.00,  0.06, -0.08,  0.04,  0.04],
                     [-0.26, -0.14,  0.06,  1.00,  0.85,  0.85,  0.85],
                     [ 0.19,  0.31, -0.08,  0.85,  1.00,  0.85,  0.85],
                     [-0.25,  0.16,  0.04,  0.85,  0.85,  1.00,  0.85],
                     [-0.12,  0.09,  0.04,  0.85,  0.85,  0.85,  1.00]])

def gen_higham_strabic(n=500):
    G = np.ones((n, n)) * 0.5; np.fill_diagonal(G, 1.0)
    G[0, n-1] = G[n-1, 0] = -0.9
    return G

def gen_higham_ex31():
    return np.array([[1.0, 1.0, 0.0],
                     [1.0, 1.0, 1.0],
                     [0.0, 1.0, 1.0]])

def gen_higham_ex33(n=100):
    G = np.eye(n) + 1.1 * np.ones((n, n)) / n
    np.fill_diagonal(G, 1.0)
    return G

def gen_qi_sun_55(n=500, seed=42):
    np.random.seed(seed)
    G = np.random.randn(n, n); C = G @ G.T
    d = np.diag(1.0 / np.sqrt(np.diag(C))); C = d @ C @ d
    R = np.random.uniform(-1, 1, (n, n)); R = (R + R.T) / 2.0
    return C + 1.0 * R

def gen_qi_sun_56(n=500, seed=42):
    np.random.seed(seed)
    G = np.random.uniform(-1, 1, (n, n)); G = (G + G.T) / 2.0
    np.fill_diagonal(G, 1.0)
    return G

def gen_minabutdinov(n=500):
    G = np.eye(n) - 10.0 * np.ones((n, n)) / n
    np.fill_diagonal(G, 1.0)
    return G

def gen_grubisic_pietersz(n=100, seed=42):
    np.random.seed(seed); r = 5
    F = np.random.randn(n, r); G = F @ F.T
    d = np.diag(1.0 / np.sqrt(np.diag(G))); G = d @ G @ d
    G += 0.05 * np.random.randn(n, n)
    return proj_U((G + G.T) / 2.0)

def gen_russell_3000(n=1000, seed=42):
    np.random.seed(seed)
    G = np.random.randn(n, 10) @ np.random.randn(10, n)
    G = (G + G.T) / 2.0
    return proj_U(G)

def gen_custom_extreme(n=200, seed=42):
    np.random.seed(seed)
    G = np.random.uniform(-2e4, 2e4, (n, n)); G = (G + G.T) / 2.0
    np.fill_diagonal(G, 1.0)
    return G

def gen_thesis_mcar(n_stocks=550, n_days=1000, missing_rate=0.25, seed=42):
    np.random.seed(seed)
    factors  = np.random.randn(10, n_days)
    loadings = np.random.randn(n_stocks, 10)
    returns  = loadings @ factors + np.random.randn(n_stocks, n_days) * 0.5
    mask     = np.random.rand(n_stocks, n_days) > missing_rate
    observed = np.where(mask, returns, np.nan)
    df       = pd.DataFrame(observed.T)
    G        = df.corr().fillna(0).values
    G        = (G + G.T) / 2.0
    np.fill_diagonal(G, 1.0)
    return G

## 4 — Sweep Configuration

In [ ]:
LM_VALUES = [0.50, 1.00, 1.95, 2.50, 3.00, 4.00, 5.00, 7.50, 10.0, 15.0, 20.0]

print("Generating all 14 test matrices...", flush=True)
SWEEP_TESTS = [
    ("1.  Rebonato-J n100",  gen_rebonato_jackel()),     # n=100
    ("2.  Turkay n4",        gen_turkay()),               # n=4
    ("3.  Bhansali n5",      gen_bhansali()),             # n=5
    ("4.  Finger n7",        gen_finger()),               # n=7
    ("5.  H-S n500",         gen_higham_strabic()),       # n=500
    ("6.  Higham Ex3.1",     gen_higham_ex31()),          # n=3
    ("7.  Higham Ex3.3",     gen_higham_ex33()),          # n=100
    ("8.  QS5.5 n500",       gen_qi_sun_55()),            # n=500
    ("9.  QS5.6 n500",       gen_qi_sun_56()),            # n=500
    ("10. Minab n500",       gen_minabutdinov()),         # n=500
    ("11. Grubisic n100",    gen_grubisic_pietersz()),    # n=100
    ("12. Russell n1000",    gen_russell_3000()),         # n=1000
    ("13. Extreme n200",     gen_custom_extreme()),       # n=200
    ("14. MCAR n550",        gen_thesis_mcar()),          # n=550
]
print(f"  {len(SWEEP_TESTS)} matrices ready.")
for name, G in SWEEP_TESTS:
    lmin = float(np.min(la.eigvalsh(G)))
    print(f"    {name:<24} n={G.shape[0]:>4}   lam_min={lmin:+.4e}")

## 5 — Printing Helpers

In [ ]:
def _term_width(default=160):
    try:
        return shutil.get_terminal_size((default, 24)).columns
    except Exception:
        return default


def _chunks(seq, k):
    for i in range(0, len(seq), k):
        yield seq[i:i + k]


# ------------------------------------------------------------------
# TABLE A: iteration counts
# ------------------------------------------------------------------
def print_iteration_table(results, higham_iters):
    name_w = max(22, max(len(n) for n, _ in SWEEP_TESTS))
    col_w = 7
    hi_w = 7
    total_w = name_w + 3 + col_w * len(LM_VALUES) + 3 + hi_w

    SEP = "=" * total_w
    MID = "-" * total_w

    print(f"\n{SEP}")
    print("TABLE A: ITERATION COUNTS  (beta=0.5, tol=1e-5, max_iter=2000)")
    print("  *  = best SPID in row   ** = best lam_max overall   DNF = 2000")
    print(SEP)

    hdr = f"{'Test':<{name_w}} |"
    hdr += "".join(f"{lm:>{col_w}.2f}" for lm in LM_VALUES)
    hdr += f" | {'Higham':>{hi_w}}"
    print(hdr)
    print(MID)

    totals = {lm: 0 for lm in LM_VALUES}
    h_total = 0

    for name, _ in SWEEP_TESTS:
        row_iters = [results[(name, lm)]["iters"] for lm in LM_VALUES]
        best = min(row_iters)
        row = f"{name:<{name_w}} |"
        for lm, it in zip(LM_VALUES, row_iters):
            tag = "*" if it == best else " "
            row += f"{it:>{col_w - 1}}{tag}"
            totals[lm] += it
        h = higham_iters[name]
        h_total += h
        row += f" | {h:>{hi_w}}"
        print(row)

    print(MID)
    best_lm = min(totals, key=totals.get)
    row = f"{'TOTAL':<{name_w}} |"
    for lm in LM_VALUES:
        tag = "**" if lm == best_lm else "  "
        row += f"{totals[lm]:>{col_w - 2}}{tag}"
    row += f" | {h_total:>{hi_w}}"
    print(row)
    print(SEP)
    print(f">>> Best lam_max = {best_lm:.2f}  (TOTAL = {totals[best_lm]})")
    print(f">>> Higham TOTAL = {h_total}")
    if 5.0 in totals and 20.0 in totals:
        print(f">>> Plateau: TOTAL(5.0) = {totals[5.0]}  vs  TOTAL(20.0) = {totals[20.0]}")


# ------------------------------------------------------------------
# TABLE B: self-regulation stats (aggregated)
# ------------------------------------------------------------------
def print_selfregulation_table(results):
    W = 100
    print(f"\n{'=' * W}")
    print("TABLE B: SELF-REGULATION STATISTICS  (aggregated over all 14 benchmarks)")
    print("  dangerous = lam_k >= 2   ceiling = alpha clipped to alpha_max   fallback = |denom| < 1e-12")
    print("=" * W)
    print(f"{'lam_max':>8} | {'iters':>7} | "
          f"{'dang':>5} {'%dang':>7} | "
          f"{'ceil':>5} {'%ceil':>7} | "
          f"{'fall':>5} {'%fall':>7}")
    print("-" * W)

    for lm in LM_VALUES:
        ti = td = tc = tf = 0
        for name, _ in SWEEP_TESTS:
            r = results[(name, lm)]
            ti += r["iters"]
            td += r["dangerous"]
            tc += r["ceiling"]
            tf += r["fallback"]

        pd_ = 100 * td / ti if ti else 0
        pc  = 100 * tc / ti if ti else 0
        pf  = 100 * tf / ti if ti else 0

        print(f"{lm:>8.2f} | {ti:>7} | {td:>5} {pd_:>6.1f}% | "
              f"{tc:>5} {pc:>6.1f}% | {tf:>5} {pf:>6.1f}%")

    print("=" * W)


# ------------------------------------------------------------------
# TABLE B2: per-problem breakdown at a given lam_max
# ------------------------------------------------------------------
def print_per_problem_stats(results, lm=5.0):
    W = 80
    print(f"\n{'-' * W}")
    print(f"TABLE B2: PER-PROBLEM BREAKDOWN at lam_max = {lm}")
    print(f"{'-' * W}")
    print(f"{'Test':<24} | {'iters':>6} | {'dang':>5} {'%':>6} | "
          f"{'ceil':>5} {'%':>6} | {'fall':>5} {'%':>6}")
    print(f"{'-' * W}")

    ti_all = td_all = tc_all = tf_all = 0
    for name, _ in SWEEP_TESTS:
        r = results[(name, lm)]
        it = r["iters"]
        d, c, f = r["dangerous"], r["ceiling"], r["fallback"]
        pd_ = 100 * d / it if it else 0
        pc  = 100 * c / it if it else 0
        pf  = 100 * f / it if it else 0
        conv_tag = "" if r["converged"] else " DNF"
        print(f"{name:<24} | {it:>6}{conv_tag:<4}| {d:>5} {pd_:>5.1f}% | "
              f"{c:>5} {pc:>5.1f}% | {f:>5} {pf:>5.1f}%")
        ti_all += it; td_all += d; tc_all += c; tf_all += f

    print(f"{'-' * W}")
    pd_ = 100 * td_all / ti_all if ti_all else 0
    pc  = 100 * tc_all / ti_all if ti_all else 0
    pf  = 100 * tf_all / ti_all if ti_all else 0
    print(f"{'TOTAL':<24} | {ti_all:>6}    | {td_all:>5} {pd_:>5.1f}% | "
          f"{tc_all:>5} {pc:>5.1f}% | {tf_all:>5} {pf:>5.1f}%")
    print(f"{'-' * W}")


# ------------------------------------------------------------------
# TABLE C: feasibility
# ------------------------------------------------------------------
def print_feasibility_table(results):
    name_w = max(24, max(len(n) for n, _ in SWEEP_TESTS))
    cell_w = 20
    term_w = _term_width()
    cols_fit = max(1, (term_w - (name_w + 3) - 1) // cell_w)

    top = "=" * min(term_w, 200)
    print(f"\n{top}")
    print("TABLE C: FEASIBILITY  (diag_err / min_eig / status)")
    print("  OK = converged   DNF = hit max_iter")
    print(top)

    for block in _chunks(LM_VALUES, cols_fit):
        hdr = f"{'Test':<{name_w}} |" + "".join(
            f"{'lam=' + f'{lm:.2f}':^{cell_w}}" for lm in block
        )
        rule = "-" * len(hdr)
        print(hdr)
        print(rule)

        for name, _ in SWEEP_TESTS:
            row = f"{name:<{name_w}} |"
            for lm in block:
                r = results[(name, lm)]
                st = "OK" if r["converged"] else "DNF"
                cell = f"{r['diag_err']:.0e}/{r['min_eig']:+.0e}/{st}"
                row += f"{cell:^{cell_w}}"
            print(row)

        print(rule)

## 6 — Run Sweep

In [ ]:
# --- Higham baseline ---
print("Computing Higham baseline for all 14 matrices...", flush=True)
higham_iters = {}
for name, G in SWEEP_TESTS:
    t0 = time.perf_counter()
    higham_iters[name] = higham_dykstra_iters(G)
    dt = time.perf_counter() - t0
    tag = "DNF" if higham_iters[name] >= 2000 else f"{higham_iters[name]}"
    print(f"  {name:<24} Higham: {tag:>5} iters  ({dt:.1f}s)", flush=True)

# --- SPID v4 sweep ---
print(f"\nRunning SPID v4 sweep: {len(SWEEP_TESTS)} matrices x "
      f"{len(LM_VALUES)} lam_max values = {len(SWEEP_TESTS)*len(LM_VALUES)} runs",
      flush=True)
results = {}
for name, G in SWEEP_TESTS:
    t0 = time.perf_counter()
    for lm in LM_VALUES:
        results[(name, lm)] = spid_v4_sweep(G, lambda_max=lm)
    dt = time.perf_counter() - t0
    # Show result at lam_max=5.0 as a quick sanity check
    r5 = results[(name, 5.0)]
    tag5 = "DNF" if not r5["converged"] else f"{r5['iters']}"
    print(f"  {name:<24} lam=5: {tag5:>5} iters  "
          f"(all 11 in {dt:.1f}s)", flush=True)

In [ ]:
# --- Print all tables ---
print_iteration_table(results, higham_iters)
print_selfregulation_table(results)
print_per_problem_stats(results, lm=5.0)
print_feasibility_table(results)

print("\n" + "=" * 70)
print("CONCLUSIONS")
print("=" * 70)
print("1. Diagonal error = 0 for ALL lam_max values (Proposition 5.1).")
print("2. No divergence observed up to lam_max = 20.0.")
print("3. Dangerous steps (lam_k >= 2) confined to transient phase;")
print("   fraction plateaus for lam_max >= 5.0 (self-regulation).")
print("4. Denominator fallback rarely triggered outside k=1 warmup.")